# 🎯 Module 13 · 09 — NLP Evaluation

> **Stage:** Machine Learning → NLP  
> **Level:** Intermediate  
> **Module:** 13 — NLP Basics / 09 — NLP Evaluation  
> **Date:** 2 February 2026  
> **Previous:** 08 — Sentiment Analysis  
> **Next:** 10 — Advanced Classical NLP

---

Building an NLP model is only **half** the problem. The other half is **measuring how well it actually performs on unseen text**.

In this notebook we'll cover:

- Why evaluation matters
- Train / Validation / Test splits
- **Accuracy, Precision, Recall, F1**
- **Confusion matrix**
- **Macro / Micro / Weighted** averages
- **Class imbalance**
- **Cross-validation** & StratifiedKFold
- **ROC-AUC** and **Precision-Recall** curves
- **NLP-specific** evaluation
- **Error analysis**
- **Data & vocabulary leakage**
- Baseline vs model comparison
- Reproducibility
- Evaluation in production

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain why a single metric is never enough
- Use **accuracy, precision, recall, F1** correctly
- Read and interpret a **confusion matrix**
- Distinguish **macro / micro / weighted** averages
- Handle **class imbalance**
- Use **cross-validation** and **StratifiedKFold**
- Explain **ROC-AUC** and **PR-AUC**
- Match **metrics to NLP tasks**
- Perform **error analysis**
- Detect and prevent **data / vocabulary leakage**
- Compare models fairly
- Ensure **reproducibility**
- Think about **production evaluation**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
)

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. Why Evaluation Matters

Consider a classifier with this class distribution:

```text
Positive → 95%
Negative →  5%
```

Predicting **Positive** for everything gives:

```text
95% accuracy
```

But it's **useless** for finding negative examples.

> **Never judge an NLP model with a single metric.**

The right evaluation depends on:

- Dataset
- Task
- Class distribution
- Business requirement
- **Cost of different errors**

---
## ✂️ 2. Train / Validation / Test Sets

```text
Dataset
   ↓
Training Set
Validation Set
Test Set
```

| Set | Purpose |
| --- | --- |
| **Training** | Learn patterns |
| **Validation** | Model selection, tuning |
| **Test** | Final evaluation on unseen data |

> The **test set should not repeatedly influence** model decisions.

In [ ]:
# Build a small synthetic sentiment dataset for full evaluation demo
positive = [
    "I love this product", "Amazing quality", "Absolutely wonderful",
    "Best purchase ever", "Highly recommend", "Fantastic experience",
    "Great service", "Excellent quality", "Very happy with this",
    "Exceeded expectations", "Really enjoyed it", "Superb product",
    "Top notch", "Really good", "Perfect for me",
]
negative = [
    "Terrible quality", "Absolutely awful", "Worst purchase ever",
    "Do not recommend", "Horrible experience", "Bad service",
    "Very disappointed", "Waste of money", "Poor quality",
    "Below expectations", "Really disliked it", "I hate it",
    "Broken on arrival", "Really bad", "Not worth it",
]

texts  = positive + negative
labels = ['positive'] * len(positive) + ['negative'] * len(negative)

df = pd.DataFrame({'text': texts, 'label': labels})
print('Dataset size:', len(df))
print(df['label'].value_counts())

In [ ]:
# Stratified split: 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label'],
)

print(f'Train: {len(X_train)}  Test: {len(X_test)}')
print('\nTrain distribution:')
print(pd.Series(y_train).value_counts(normalize=True).round(3))
print('\nTest distribution:')
print(pd.Series(y_test).value_counts(normalize=True).round(3))

---
## 📈 3. Accuracy

$$\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}$$

```python
from sklearn.metrics import accuracy_score
accuracy_score(y_test, predictions)
```

> Accuracy is useful only when classes are **reasonably balanced**.

---
## 🔢 4. Confusion Matrix

```text
                  Predicted
                Positive  Negative

Actual Positive    TP        FN
Actual Negative    FP        TN
```

| Term | Meaning |
| --- | --- |
| **TP** | True Positive |
| **TN** | True Negative |
| **FP** | False Positive |
| **FN** | False Negative |

Example:

```text
                  Predicted
                Pos      Neg

Actual Pos      80        10
Actual Neg       5       105
```

This tells more than accuracy alone.

---
## 🎯 5. Precision

> When the model predicts positive, how often is it correct?

$$\text{Precision} = \frac{TP}{TP + FP}$$

High precision → **few false positives**.

Important when **false positives are costly**.

---
## 🔍 6. Recall

> Of all actual positive examples, how many did the model find?

$$\text{Recall} = \frac{TP}{TP + FN}$$

High recall → **few false negatives**.

Important when **missing positives is costly**.

---
## ⚖️ 7. F1-Score

$$F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Balances precision and recall.

```python
from sklearn.metrics import f1_score
f1_score(y_test, predictions, average='binary')
```

For **multiclass**, choose `average='macro' | 'micro' | 'weighted'`.

---
## 🧪 8. Precision, Recall & F1 Example

Consider two models:

| Model | Precision | Recall |
| --- | --- | --- |
| A | 0.95 | 0.20 |
| B | 0.75 | 0.75 |

Model A looks excellent on precision but misses **most positives**.  
Model B is more balanced.

> The right choice depends on the **task** and **cost of errors**.

---
## 📋 9. Classification Report

```python
from sklearn.metrics import classification_report
print(classification_report(y_test, predictions))
```

Output:

```text
              precision    recall    f1-score
negative         0.84       0.80       0.82
positive         0.87       0.90       0.88
accuracy                               0.86
```

Gives **class-level** performance — not just one number.

---
## 📊 10. Macro, Micro, Weighted Averages

### Macro
Average metric per class, **equal weight per class**.
Useful when **every class matters equally** — good with class imbalance.

### Weighted
Weight each class by its **number of examples**.
Larger classes dominate the average.

### Micro
Aggregate across all classes **before** computing the metric.
Close to accuracy in single-label multiclass.

Pick the average that matches your task.

---
## ⚖️ 11. Class Imbalance

```text
Positive → 9000
Negative → 1000
```

High accuracy → useless minority-class performance.

### Inspect

- Class distribution
- Confusion matrix
- Precision
- Recall
- F1

> **Macro** metrics prevent large classes from dominating.

---
## 🔄 12. Cross-Validation

Instead of one train/test split, use **K-fold** cross-validation:

```text
Fold 1 → validation
Fold 2 → validation
...
Fold K → validation
```

```python
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5, scoring='f1_macro')
```

Gives a **more stable estimate** of performance.

---
## 🧬 13. StratifiedKFold

Preserves class proportions across folds — critical for **classification** and **imbalanced data**.

```python
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
```

---
## 📈 14. ROC-AUC

Evaluates the classifier across **all decision thresholds**.

Plots:
```text
True Positive Rate
        vs
False Positive Rate
```

`AUC` = Area Under the Curve.

```python
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, probabilities)
```

> On **heavily imbalanced** data, **PR-AUC is often more informative** than ROC-AUC.

---
## 📐 15. Precision-Recall Curve

Shows **precision vs recall** at different thresholds.

Useful when the positive class is **rare**:

- Spam detection
- Fraud detection
- Rare intent detection
- Toxic content detection

> Different thresholds → different precision/recall trade-offs.

---
## 📚 16. NLP-Specific Evaluation

Different tasks → different metrics.

| Task | Typical metrics |
| --- | --- |
| Text classification | Accuracy, Precision, Recall, F1, Confusion matrix |
| Sentiment analysis | Accuracy, Macro F1, Weighted F1, Confusion matrix |
| Similarity | Cosine similarity, Correlation, Ranking metrics |
| Information retrieval | Precision@K, Recall@K, MRR, MAP, NDCG |
| Sequence tasks | Token accuracy, F1 (per token/entity) |

> Evaluation should always match the **task**.

---
## 🔍 17. Error Analysis

Metrics tell you **how much** the model fails.  
Error analysis tells you **why**.

Build a table:

| Text | Actual | Predicted | Error Type |
| --- | --- | --- | --- |
| not bad | Positive | Negative | Negation |
| amazing... broken again | Negative | Positive | Sarcasm |
| phone bohat acha hai | Positive | Negative | Roman Urdu |
| battery okay but camera terrible | Negative | Positive | Mixed sentiment |

---
## 🗂️ 18. Common NLP Error Categories

- **Negation** — `"not good"`
- **Sarcasm** — `"Great, another problem."`
- **Ambiguity** — `"This is sick."`
- **Spelling errors** — `"amaazing"`
- **Slang** — `"this movie slaps"`
- **Code-switching** — `"ye product really acha hai"`
- **Domain-specific** language

Categorizing errors helps you decide what to fix.

---
## 🚫 19. Data Leakage

### ❌ Wrong

```text
Entire Dataset
     ↓
Fit TF-IDF
     ↓
Train/Test Split
```

### ✅ Correct

```text
Dataset
   ↓
Train/Test Split
   ↓
Fit TF-IDF on Training Data
   ↓
Transform Test Data
   ↓
Evaluate
```

Use a **Pipeline** to enforce this automatically.

---
## 📖 20. Vocabulary Leakage

### ❌ Wrong

```python
vectorizer.fit(all_text)
# then split
```

The vectorizer **has seen** test data.

### ✅ Correct

```python
vectorizer.fit(X_train)
X_tr = vectorizer.transform(X_train)
X_te = vectorizer.transform(X_test)
```

Pipeline handles this automatically.

---
## 📏 21. Baseline Comparison

Always establish a **baseline** first.

```text
Baseline
   ↓
TF-IDF + Naive Bayes
   ↓
TF-IDF + Logistic Regression
   ↓
TF-IDF + Linear SVM
```

Example comparison table:

| Model | Accuracy | Macro F1 |
| --- | ---: | ---: |
| Baseline | 0.70 | 0.62 |
| Naive Bayes | 0.81 | 0.78 |
| Logistic Regression | 0.84 | 0.82 |
| Linear SVM | 0.85 | 0.83 |

> Compare models under **identical conditions**.

---
## 🆚 22. Model Comparison

Keep the evaluation **consistent**:

```text
Same dataset
Same test set
Same preprocessing
Same metrics
```

Then compare — otherwise conclusions are unreliable.

---
## 🎛️ 23. Hyperparameter Tuning & Evaluation

Use metrics during tuning — but **never** on the test set.

```python
grid = GridSearchCV(model, param_grid, cv=5, scoring='f1_macro')
grid.fit(X_train, y_train)
```

Test set stays **untouched** for the final evaluation.

---
## 🧪 24. Full Evaluation Demo

We'll build a pipeline, evaluate it, and inspect metrics.

In [ ]:
# Build a baseline pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

pipe.fit(X_train, y_train)
preds = pipe.predict(X_test)
probs = pipe.predict_proba(X_test)[:, 1]  # probability of class 'positive'

print('Accuracy :', round(accuracy_score(y_test, preds), 3))
print('Precision:', round(precision_score(y_test, preds, pos_label='positive'), 3))
print('Recall   :', round(recall_score(y_test, preds, pos_label='positive'), 3))
print('F1       :', round(f1_score(y_test, preds, pos_label='positive'), 3))

In [ ]:
# Full classification report
print(classification_report(y_test, preds, digits=3))

In [ ]:
# Confusion matrix heatmap
labels_order = pipe.classes_
cm = confusion_matrix(y_test, preds, labels=labels_order)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels_order)))
ax.set_yticks(range(len(labels_order)))
ax.set_xticklabels(labels_order)
ax.set_yticklabels(labels_order)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
for i in range(len(labels_order)):
    for j in range(len(labels_order)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=13)
plt.colorbar(im)
plt.title('Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# ROC-AUC and PR-AUC
auc = roc_auc_score((y_test == 'positive').astype(int), probs)
ap = average_precision_score((y_test == 'positive').astype(int), probs)

print(f'ROC-AUC : {auc:.3f}')
print(f'PR-AUC  : {ap:.3f}')

In [ ]:
# Plot ROC and PR curves side by side
y_bin = (y_test == 'positive').astype(int)
fpr, tpr, _ = roc_curve(y_bin, probs)
prec, rec, _ = precision_recall_curve(y_bin, probs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(fpr, tpr, label=f'AUC = {auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(rec, prec, label=f'AP = {ap:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## 🔄 25. Cross-Validation Demo

In [ ]:
# 5-fold StratifiedKFold cross-validation on training data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipe, X_train, y_train,
    cv=cv, scoring='f1_macro', n_jobs=-1,
)

print('Per-fold F1 (macro):', scores.round(3))
print(f'Mean                : {scores.mean():.3f}')
print(f'Std                 : {scores.std():.3f}')

---
## ⚖️ 26. Class Imbalance Demo

Compare **default** vs **balanced** class weights.

In [ ]:
# Build an imbalanced dataset (90% positive, 10% negative)
imb_texts  = positive * 9 + negative * 1
imb_labels = ['positive'] * (len(positive) * 9) + ['negative'] * len(negative)

Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(
    imb_texts, imb_labels,
    test_size=0.25, random_state=42, stratify=imb_labels,
)

for cw in [None, 'balanced']:
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
        ('clf', LogisticRegression(max_iter=1000, class_weight=cw, random_state=42)),
    ])
    pipe.fit(Xtr_i, ytr_i)
    p = pipe.predict(Xte_i)
    tag = 'default  ' if cw is None else 'balanced '
    print(f'{tag} acc={accuracy_score(yte_i, p):.3f}  '
          f'f1(macro)={f1_score(yte_i, p, average="macro"):.3f}  '
          f'recall(negative)={recall_score(yte_i, p, pos_label="negative"):.3f}')

---
## 🆚 27. Model Comparison Demo

In [ ]:
# Compare NB, LR, and LinearSVC under identical conditions
classifiers = {
    'Naive Bayes': MultinomialNB(),
    'Logistic':    LogisticRegression(max_iter=1000, random_state=42),
    'LinearSVC':   LinearSVC(random_state=42),
}

print(f'{"Model":<14} {"Acc":>8} {"Macro F1":>10} {"Weighted F1":>12}')
print('-' * 48)
for name, clf in classifiers.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
        ('clf', clf),
    ])
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test)
    print(f'{name:<14} '
          f'{accuracy_score(y_test, p):>8.3f} '
          f'{f1_score(y_test, p, average="macro"):>10.3f} '
          f'{f1_score(y_test, p, average="weighted"):>12.3f}')

---
## 🔍 28. Error Analysis Demo

In [ ]:
# Show wrong predictions from the baseline pipeline
wrong = [(t, a, p) for t, a, p in zip(X_test, y_test, preds) if a != p]

print(f'Test samples    : {len(X_test)}')
print(f'Misclassified   : {len(wrong)}')
print()
if wrong:
    print('Examples of wrong predictions:')
    for text, actual, pred in wrong[:8]:
        print(f'  {text!r}')
        print(f'    actual={actual}  predicted={pred}')
else:
    print('No misclassifications on this tiny test set.')

---
## 🎛️ 29. Hyperparameter Tuning Demo

In [ ]:
# Small grid search on the sentiment dataset
pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

params = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df':      [1, 2],
    'clf__C':             [0.1, 1, 10],
}

grid = GridSearchCV(pipe, params, cv=3, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print(f'Best CV F1 : {grid.best_score_:.3f}')
print(f'Test F1    : {f1_score(y_test, grid.predict(X_test), average="macro"):.3f}')

---
## 🧠 30. Mental Model

```text
TRAIN
  ↓
VALIDATE
  ↓
TEST
  ↓
MEASURE
  ↓
UNDERSTAND ERRORS
  ↓
IMPROVE
  ↓
RE-EVALUATE
```

### Main metrics

```text
Accuracy  → How many predictions were correct?
Precision → When I predict positive, how often am I right?
Recall    → How many actual positives did I find?
F1        → How well are precision and recall balanced?
Confusion Matrix → What types of mistakes are happening?
```

---
## 🏋️ 31. Hands-On Exercises

### Exercise 1 — Metrics by hand
Given `TP=80, FP=15, FN=5, TN=100`, compute:
- Accuracy
- Precision
- Recall
- F1

Verify with scikit-learn.

### Exercise 2 — Confusion matrix reading
Given a confusion matrix, identify which class the model struggles with most. What metric would you use?

### Exercise 3 — Macro vs weighted F1
Create an imbalanced dataset. Compare **macro F1** and **weighted F1**. Which better reflects minority-class performance?

### Exercise 4 — Cross-validation
Run 5-fold **StratifiedKFold** on a classifier. Report mean and std of `f1_macro`.

### Exercise 5 — ROC vs PR
On an imbalanced dataset, compute both ROC-AUC and PR-AUC. Which is more informative?

### Exercise 6 — Error analysis
Take a trained model's misclassified examples. Group them into categories (negation, sarcasm, slang, etc.). Which is the dominant failure mode?

### Exercise 7 — Data leakage
Write two code snippets — one that leaks (fit vectorizer on all data), and one that doesn't. Compare the reported F1.

---
## 🎤 32. Interview Questions

1. **Why isn't accuracy enough?**  
   Imbalanced data can produce high accuracy with useless minority-class performance.

2. **Train / validation / test?**  
   Train = learn; validation = tune / select; test = final evaluation.

3. **What is a confusion matrix?**  
   A table of TP, TN, FP, FN — shows types of mistakes.

4. **Precision vs recall?**  
   Precision = TP/(TP+FP). Recall = TP/(TP+FN).

5. **When is precision more important?**  
   When **false positives are costly**.

6. **When is recall more important?**  
   When **false negatives are costly**.

7. **What is F1?**  
   Harmonic mean of precision and recall.

8. **Macro vs weighted average?**  
   Macro = equal weight per class; weighted = weighted by class size.

9. **What is cross-validation?**  
   Multiple train/val splits to get a more reliable performance estimate.

10. **What is StratifiedKFold?**  
    K-fold that preserves class proportions across folds.

11. **What is ROC-AUC?**  
    Area under the ROC curve — measures ranking across all thresholds.

12. **When to prefer PR-AUC over ROC-AUC?**  
    On heavily imbalanced datasets.

13. **What is data leakage?**  
    When test information influences training — inflating performance.

14. **What is vocabulary leakage?**  
    Fitting the vectorizer on all data (including test) before splitting.

15. **How do you compare models fairly?**  
    Same data, split, preprocessing, and metrics.

16. **Why is error analysis important?**  
    It reveals **why** a model fails — categories, not just counts.

17. **Why is reproducibility important?**  
    To verify, compare, and debug results reliably.

18. **How does production evaluation differ?**  
    It's **continuous** — monitor data drift, collect feedback, retrain.

---
## 🏁 33. Key Takeaways

### You now understand

- Why NLP models need evaluation
- Train / validation / test
- Stratification
- Accuracy, precision, recall, F1
- Confusion matrix
- Macro / micro / weighted averages
- Class imbalance
- Cross-validation & StratifiedKFold
- ROC-AUC and PR curves
- Data & vocabulary leakage
- Baseline vs model comparison
- Hyperparameter tuning and evaluation
- Error analysis
- NLP-specific evaluation
- Reproducibility
- Production evaluation

### 🧭 Golden Rule

> **Don't just ask whether the model is accurate. Ask which examples it gets wrong, why it gets them wrong, and whether those errors matter for the real application.**

---

### 🔗 What's Next?

**10 — Advanced Classical NLP**

Topic modeling, NER, information retrieval, clustering, and a complete end-to-end classical NLP workflow.